In [1]:
import sys
import os
project_root = "/Users/linarojas/Desktop/Research/Papers/Combinatorial_Ternary/Hybrid-Experimental-Data-Driven-Workflow"
sys.path.append(sys.path.append(project_root))
path_root = "/Users/linarojas/Desktop/Research/Papers/Combinatorial_Ternary/Hybrid-Experimental-Data-Driven-Workflow"

In [2]:
from Data_Extraction.database_manager import DatabaseManager
from Data_Extraction.DataPreprocessor import DataPreprocessor

import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

Databases
1. Syntetic data EMA: Alloys properties determined using the effective medium approximation
2. Pure elements: Properties of the pure metals (Cu, Ni, Al) measured experimentally
3. Ternary round 1: Properties of the alloys measured experimentally, the composition was determined applying the experimental design for mixture Centroid 

In [4]:
db = DatabaseManager(path_root+"/Data_Extraction/Databases/Synthetic_data_EMA.db")
tables = db.list_tables()
print (tables)
print(db.col_names("compositions"))
print(db.col_names("Drude_model"))
print(db.col_names("Optical_properties"))
df_EMA_data = db.table_dataframe(table_name='Optical_properties')
db.close()

Connected to database: /Users/linarojas/Desktop/Research/Papers/Combinatorial_Ternary/Hybrid-Experimental-Data-Driven-Workflow/Data_Extraction/Databases/Synthetic_data_EMA.db
['Drude_model', 'compositions', 'Optical_properties']
Colum names: ['ID', 'Cu', 'Ni', 'Al']
None
Colum names: ['Cu', 'Ni', 'Al', 'Resistivity', 'Relax_time', 'ID']
None
Colum names: ['Cu', 'Ni', 'Al', 'wavelength_nm', 'e1', 'e2', 'ID']
None
Database connection closed


In [5]:
db = DatabaseManager(path_root+"/Data_Extraction/Databases/Pure_elements.db")
tables = db.list_tables()
print (tables)
print(db.col_names("compositions"))
print(db.col_names("optical_properties"))
print(db.col_names("lorentz"))
db.close()

Connected to database: /Users/linarojas/Desktop/Research/Papers/Combinatorial_Ternary/Hybrid-Experimental-Data-Driven-Workflow/Data_Extraction/Databases/Pure_elements.db
['compositions', 'sqlite_sequence', 'optical_properties', 'lorentz']
Colum names: ['ID', 'Cu', 'Ni', 'Al', 'Resistivity', 'Relax_time', 'Thickness_nm']
None
Colum names: ['OP_ID', 'ID', 'wavelength_nm', 'e1', 'e2']
None
Colum names: ['lorentz_ID', 'ID', 'Amp', 'Br', 'En']
None
Database connection closed


In [6]:
db = DatabaseManager(path_root+"/Data_Extraction/Databases/Ternary_round1.db")
tables = db.list_tables()
print (tables)
print(db.col_names("compositions"))
print(db.col_names("Thickness"))
print(db.col_names("Optical_properties"))
db.close()

Connected to database: /Users/linarojas/Desktop/Research/Papers/Combinatorial_Ternary/Hybrid-Experimental-Data-Driven-Workflow/Data_Extraction/Databases/Ternary_round1.db
['compositions', 'Thickness', 'Optical_properties']
Colum names: ['ID', 'Cu', 'Ni', 'Al']
None
Colum names: ['ID', 'Thickness']
None
Colum names: ['wavelength_nm', 'e1', 'e2', 'ID']
None
Database connection closed


Simulated Data Structure
1. Electronic distribution of the elements of the periodic table
2. Simulations from AFLOW, components, stoichiometry, thermal conductivity
3. Extract the data AFLOW and convert it into a Dataframe (Drop Thermal Conductivity)
4. Organize the dataframe to display the element and the composition


In [7]:
# 1. Electronic distribution of the periodic table

elec_struc = os.path.join(
    project_root,
    "Data_Extraction",
    "Electronic_configuration.pkl"
)


with open(elec_struc,"rb") as f:
    elec_data = pickle.load(f)

In [8]:
#2.Simulations from AFLOW
dataf_aflow = os.path.join(
    project_root,
    "Data_Extraction",
    "Data_Extraction_AFLOW.pkl"
)
data_aflow = pd.read_pickle(dataf_aflow)

In [9]:
#3. Extract the data AFLOW and convert it into a Dataframe (Drop Thermal Conductivity)
keys = data_aflow.keys()

simulation_data = []

# Unifying the Dataframes
for key, df in data_aflow.items():
    subset = df[['species', 'stoichiometry', 'agl_thermal_conductivity_300K']].copy()
    subset['source'] = key   # optional: label where it came from
    simulation_data.append(subset)

simulation_data = pd.concat(simulation_data, ignore_index=True)
simulation_data.rename(columns={'agl_thermal_conductivity_300K':'thermal_conductivity'}, inplace=True)

## Splitting the Species and stoichiometry data
simulation_data["species_list"] = simulation_data["species"].str.replace(" ","").str.split(",")
simulation_data["stoich_list"] = simulation_data["stoichiometry"].str.replace(" ","").str.split(",")

In [14]:
# 4. Organize the dataframe to display the element and the composition
unique_elements = simulation_data['species_list'].explode().unique().tolist()
unique_elements.append('Thermal_conductivity') # Adding the thermal conductivity column

df_composition = pd.DataFrame(columns=unique_elements)

## Build rows
row = []

for species_list,stoich_list,thermal in zip (simulation_data['species_list'],
                                             simulation_data['stoich_list'],
                                             simulation_data['thermal_conductivity']):
    row_dict = {}
    
    # Fill composition for each specie
    for element, compo in zip(species_list,stoich_list):
        row_dict[element] = compo
        df_composition[element] = compo
    
    # Fill missing species with 0
    for elem in unique_elements:
        if elem not in row_dict and elem != 'Thermal_conductivity':
            row_dict[elem] = 0
    
    # Add thermal conductivity
    #row_dict['Thermal_conductivity'] = thermal
    row.append(row_dict)
    
df_composition = pd.DataFrame(row)

Dimensionality Reduction Electronic Configuration

1. Flat the electronic distribution matrix
2. TSNE dimensionality reduction

In [16]:
# 2. Flat the electronic distribution matrix
def flat_matrix(elec_data: dict, pad_value=0.0):

    elements = sorted([k for k, v in elec_data.items() if v is not None])

    mats = [np.asarray(elec_data[el], dtype=float) for el in elements]
    shapes = mats[0].shape

    rows = shapes[0]
    colm = shapes[1]

    X = np.full((len(elements), rows, colm), pad_value, dtype=float)
    for i, m in enumerate(mats):
        r, c = m.shape
        X[i, :r, :c] = m

    X_flat = X.reshape(len(elements), -1)
    return elements, X_flat

In [17]:
# 3. TSNE dimensionality reduction

def tsne_elec_data(elec_data: dict, n_components=2, perplexity=20, learning_rate="auto", random_state=42):
    elements, X_flat = flat_matrix(elec_data)

    scaler = StandardScaler()
    Xs = scaler.fit_transform(X_flat)

    tsne = TSNE(
        n_components=n_components,
        perplexity=perplexity,  
        learning_rate=learning_rate,
        init="pca",
        random_state=random_state,
    )
    Z = tsne.fit_transform(Xs)

    df_embed = pd.DataFrame(Z, columns=[f"TSNE{i+1}" for i in range(n_components)])
    df_embed.insert(0, "element", elements)
    return df_embed, tsne, scaler

df_tsne, tsne_model, scaler = tsne_elec_data(elec_data, perplexity=20)
print(df_tsne.head())

  element     TSNE1     TSNE2
0      Ag  1.289671 -0.305420
1      Al -2.234458 -9.150499
2      As  0.290035 -4.246981
3      Au  6.986150  8.079452
4       B -4.848558 -9.859792


Final Dataframe

1. Join dataframe composition and TSNE dimensionality reduction

In [31]:
df_red = df_tsne.set_index("element") # Convert the elements names in indexes
dims = df_red.columns.tolist() 

elemnt = [el for el in X_train.columns if el in df_red.index]


X = X_train[elemnt].to_numpy(dtype=float)
Z = df_red.loc[elemnt, dims].to_numpy(dtype=float)  

W = X[:, :, None] * Z[None, :, :]
W2 = W.reshape(X.shape[0], -1)

colnames = [f"{el}_{d}" for el in elemnt for d in dims]

df_tsne_train = pd.DataFrame(W2, index=X_train.index, columns=colnames)
df_tsne_train['Thermal_conductivity'] = y_train

# Save the dataframe
df_tsne_train.to_pickle('tsne_train.pkl')

In [32]:
df_red = df_tsne.set_index("element") # Convert the elements names in indexes
dims = df_red.columns.tolist() 

elemnt = [el for el in X_test.columns if el in df_red.index]


X = X_test[elemnt].to_numpy(dtype=float)
Z = df_red.loc[elemnt, dims].to_numpy(dtype=float)  

W = X[:, :, None] * Z[None, :, :]
W2 = W.reshape(X.shape[0], -1)

colnames = [f"{el}_{d}" for el in elemnt for d in dims]

df_tsne_test = pd.DataFrame(W2, index=X_test.index, columns=colnames)
df_tsne_test['Thermal_conductivity'] = y_test

# Save the dataframe
df_tsne_test.to_pickle('tsne_test.pkl')